In [21]:
"""RAG PIPELINES - DATA INGESTION TO VECTOR DB PIPILINE"""

'RAG PIPELINES - DATA INGESTION TO VECTOR DB PIPILINE'

In [22]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [23]:
###Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents  

# Process all PDFs in the data direcotry
all_pdf_documents = process_all_pdfs("data")

Found 0 PDF files to process

Total documents loaded: 0


In [24]:
###Text splitting get into chunks


def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Splitting documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
        
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    ## Showing an example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [25]:
chunks=split_documents(all_pdf_documents)
chunks

Split 0 documents into 0 chunks


[]

In [26]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity



In [27]:
class EmbeddingManager:

    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """

        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings

        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:  

        """
        Generate embeddings for a list of texts

        Args: 
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
            """

        if not self.model:
            raise ValueError("Model not loaded")
    
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings 

## initilialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1254.84it/s]


Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_12241/2642550724.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore


In [28]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = 'pdf_documents', persist_directory: str = "data/vector_store"):

        """ 
        Intilaize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
            
        """
    
        self.collection_name = collection_name
        self.persist_directory = persist_directory        
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
            """Initialize ChromaDB client and collection"""
            try:
                #Create persistent ChromaDB client
                os.makedirs(self.persist_directory, exist_ok=True)
                self.client = chromadb.PersistentClient(path=self.persist_directory)

                #Get or create collection
                self.collection = self.client.get_or_create_collection(
                    name=self.collection_name,
                    metadata={"description": "PDF document embeddings for RAG"}


                )
                print(f"Vector store initialized. Collection: {self.collection_name}")
                print(f"Existing documents in collection: {self.collection.count()}")

            except Exception as e:
                print(f"Error initializing vector store: {e}")
                raise 

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
            """Add documents and their embeddings to the vector store

                args:
                    documents: List of LangChain documents
                    embeddings: Corresponding embeddings for the documents
            """
            if len(documents) != len(embeddings):
                raise ValueError("Number of documents must match the number of embeddings")
            
            print(f"Adding {len(documents)} documents to vector store...")

            # Prepre data for ChromaDB
            ids = []
            metadatas = []
            documents_text = []
            embeddings_list = []

            for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
                # Generate unique ID
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
                ids.append(doc_id)

                #Prepare metadata
                metadata = dict(doc.metadata)
                metadata['doc_index'] = i
                metadata['content_length'] = len(doc.page_content)
                metadatas.append(metadata)

                # Document content
                documents_text.append(doc.page_content)

                # Embedding
                embeddings_list.append(embedding.tolist())

            #Add to collection
            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text
                )
                print(f"Successfuly added {len(documents)} documents to vector store")
                print(f"Total documents in collection: {self.collection.count()}")

            except Exception as e:
                print(f"Error adding documents to vector store: {e}")
                raise


vectorstore=VectorStore()
vectorstore
        

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [29]:
chunks

[]

In [30]:
### Convert the text to embeddings...
texts=[doc.page_content for doc in chunks]


# Generate the embeddings

embeddings=embedding_manager.generate_embeddings(texts)

#tore in the vector db
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 0 texts...


Batches: 0it [00:00, ?it/s]

Generated embeddings with shape: (0,)
Adding 0 documents to vector store...
Error adding documents to vector store: Expected Embeddings to be non-empty list or numpy array, got [] in add.


ValueError: Expected Embeddings to be non-empty list or numpy array, got [] in add.

"""Retrieval pipeline form vector db


In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Intialize the retriever class

        Args:
        vector_store: vector store containing document embeddings
        embedding_manager: manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:

        """
        Retrieve relevant documents for a query

        Args: 
            query: the search query
            top_k: no of top results to return
            score_threshold: minimum similarity score threshold

        returns:
            List of dictionaries containing retrieved documents and metadata
            """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k

            )

            # process the results gotten from the search
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # converting distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever= RAGRetriever(vectorstore,embedding_manager)


In [ ]:
rag_retriever.retrieve("What is Prohibited  conduct")

Retrieving documents for query: 'What is Prohibited  conduct
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 75.48it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_09bb9b86_49',
  'content': 'and/or   to    the   CEO, so    th at    an   investigation    can   be   commenced where \nappropriate. Where the complaint is against the CEO, the complaint may be made to the \nBoard. \nGESCI’s policy similarly prohibits sexually harassing conduct by any GESCI personnel \nthat may create an intimidating hostile or offensive work environment whether it be in the \nform of physical, verbal or visual harassment and regardless of whether committed by an \nindividual with sup ervisory authority or by any other individual. Such conduct includes \nbut is not limited to unwelcome sexual flirtations, advances or   propositions; verbal   abuse   \nof   a   sexual   nature; graphic   verbal   comments   about an individual’s body; sexually \ndegrading words used to  describe an individu al; and the display in the  workplace of \nsexually suggestive objects or pictures. \nAny employee who believes that conduct in violation of this policy may be occurring

Integration VectorDB context pipeline with LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)

## 2. Simple  RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retrieve the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return " NO relevant context found to answer the question. "
    
    ## generating the answer using GROQ LLM
    prompt=f"""Use the folllowing context to answer the question concisely.
    Context:
    {context}

    Question: {query}

    Answer:"""

    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [ ]:
answer=rag_simple("What is Confidential  Information?",rag_retriever, llm)
print(answer)

"" embedding and vectorstoreDB